In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\emirh\AppData\Local\Temp\ipykernel_28264\3190134721.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
# 1. Yönerge PDF'ini yükleyin (Dosya adını kendi PDF'inize göre güncelleyin)
loader = PyPDFLoader(
    "tesvik_rehberi.pdf"
)
docs = loader.load()

# 2. Metni parçalara ayırın
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=50
)
splits = text_splitter.split_documents(docs)

# 3. Embedding ve Vektör Veritabanı (Chroma) oluşturun
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)  # En alakalı 3 maddeyi getir

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [3]:
# LLM modelini tanımlayın (Bilgisayarınızda yüklü olan model adını yazın, örn: llama3.2)
llm = ChatOllama(model="llama3", temperature=0)

# Denetçi / Doğrulayıcı Prompt Şablonu
template = """Sen üniversite BAP projeleri denetim ve uyumluluk uzmanısın. 
Sana sağlanan "BAP Bilişim, Donanım ve Ar-Ge Teçhizatı Tedarik ve Destek Yönergesi" maddelerine dayanarak, akademisyenin talep ettiği kalemin kurallara uygun olup olmadığını denetle.

Kurallar:
1. Talep edilen kalemin yönergeye göre durumunu kesin olarak **[UYGUN / DOĞRU]** veya **[UYGUN DEĞİL / YANLIŞ]** şeklinde belirt.
2. Kararının gerekçesini yönergedeki ilgili maddeye (örn. Madde 5, Madde 6) atıfta bulunarak net bir şekilde açıkla.

Yönerge Maddeleri (Bağlam):
{context}

Akademisyenin Talep Ettiği Kalem/Soru: {question}

Denetim Sonucu ve Gerekçesi:"""

prompt = ChatPromptTemplate.from_template(template)


# 3. Belge formatlama fonksiyonu
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)


# 4. Düzeltilmiş RAG Zinciri (RunnablePassthrough doğrudan 'question' parametresini alır)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [4]:


# 5. Test Kalemlerini Çalıştırma
test_kalemleri = [
    "Projem için yüksek performanslı ekran kartı (GPU) ve iş istasyonu alabilir miyim?",
    (
        "Günlük işlerimi yürütmek ve rapor yazmak için standart bir dizüstü"
        " (laptop) bilgisayar talep ediyorum."
    ),
    (
        "Arduino geliştirme kartları ve bağlantı kablolarını sarf malzemesi"
        " kaleminden alabilir miyim?"
    ),
    "Proje bittikten sonra aldığım sunucuları kişisel evime götürebilir miyim?",
]

for kalem in test_kalemleri:
  print(f"--- TALEP: {kalem} ---")
  cevap = rag_chain.invoke(kalem)
  print(cevap)
  print("=" * 50 + "\n")

--- TALEP: Projem için yüksek performanslı ekran kartı (GPU) ve iş istasyonu alabilir miyim? ---
Talep edilen kalemin durumunu **UYGUN** olarak belirtiyorum.

Gerekçesi: Yönerge Maddesi 1'e atıfta bulunarak, yüksek kapasiteli sunucu, GPU ve iş istasyonlarının mülkiyeti BAP Komisyonu kararıyla ilgili Bölüm/Fakülte Laboratuvarına veya Üniversite Veri Merkezi devredilir. Bu maddede, yüksek performanslı donanımların laboratuvar veya veri merkezi ortamına devredilmesi belirtilmiştir. Bu nedenle, akademisyenin talep ettiği yüksek performanslı ekran kartı (GPU) ve iş istasyonu alımı, BAP Komisyonu kararıyla ilgili laboratuvar veya veri merkezi ortamına devredilmesi gerekir.

--- TALEP: Günlük işlerimi yürütmek ve rapor yazmak için standart bir dizüstü (laptop) bilgisayar talep ediyorum. ---


KeyboardInterrupt: 

In [5]:
# Analiz edilmesini istediğiniz donanım ve malzeme listesi
donanim_listesi = [
    "Genel amaçlı standart dizüstü (laptop) bilgisayar",
    "Yapay zeka modeli eğitimi için yüksek performanslı ekran kartı (GPU)",
    "Veri işleme için fiziksel sunucu ve iş istasyonu (Workstation)",
    "Arduino geliştirme kartları ve bağlantı kabloları",
    "Günlük ofis işleri için masaüstü bilgisayar",
]

print("=== BAP DONANIM UYUMLULUK ANALİZ RAPORU ===\n")

for i, donanim in enumerate(donanim_listesi, 1):
  print(f"[{i}] İncelenen Kalem: {donanim}")

  # RAG zincirine kalemi gönderiyoruz
  cevap = rag_chain.invoke(donanim)

  print(f"Analiz Sonucu:\n{cevap}")
  print("-" * 60 + "\n")

=== BAP DONANIM UYUMLULUK ANALİZ RAPORU ===

[1] İncelenen Kalem: Genel amaçlı standart dizüstü (laptop) bilgisayar
Analiz Sonucu:
Talep edilen kalemin durumunu **UYGUN DEĞİL / YANLIŞ** olarak belirtiyorum.

Gerekçesi: Yönerge Maddesi 7'de, talep edilen kalemin "Workstation, sunucu, veri depolama birimlerini, yüksek performanslı hesaplama sistemleri, bulut bilişim hizmetleri ve mikrodenetleyici donanımlarının alım, değerlendirilmesi, bütçelenmesi, demirbaş kayıt ve kullanımına ilişkin usul ve esasları" ifade eder. Genel amaçlı standart dizüstü (laptop) bilgisayar, bu tanımlamaya uymuyor. Talep edilen kalemin, yüksek performanslı hesaplama sistemleri, bulut bilişim hizmetleri ve mikrodenetleyici donanımlarına ilişkin bir talep olmadığı için, yönergeye uygun değildir.
------------------------------------------------------------

[2] İncelenen Kalem: Yapay zeka modeli eğitimi için yüksek performanslı ekran kartı (GPU)
Analiz Sonucu:
Talep edilen kalemin durumunu **UYGUN** olarak belirtiyo